# Frutalândia — O Contador do Butico (Google Colab)

**Seis cenas × 8 segundos = 48 segundos.** Produção de animação **3D procedural real** no Blender, narração Piper e montagem FFmpeg. Tudo open source / gratuito; **nenhum serviço pago nem publicação automática**.

**Execute as células em ordem, na sua sessão Colab.** O Colab gratuito pode não conceder GPU — WORKBENCH é o modo inicial e funciona via CPU/OpenGL software com Xvfb. CYCLES é experimental e pode apresentar crash; o visual de WORKBENCH é mais simples.

As cenas terminadas permanecem na pasta `frutalandia-colab` da sessão; se uma cena falhar, é possível reexecutar somente ela. Faça backup/download antes do ambiente Colab ser encerrado.

In [ ]:
#@title 1. Configurar produção (R$ 0)
MOTOR_3D = 'WORKBENCH' #@param ['WORKBENCH', 'CYCLES']
CENAS = 'all' #@param ['all', 's01', 's02', 's03', 's04', 's05', 's06']
BAIXAR_AUTOMATICAMENTE = True #@param {type:'boolean'}

from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, time, re
PROJECT = Path('/content/moneyPrintTest')
BASE = Path('/content/frutalandia-colab/butico')
CLIPS = BASE / 'clips'
RAW = BASE / 'raw'
FINAL = BASE / 'final'
for target in (CLIPS, RAW, FINAL): target.mkdir(parents=True, exist_ok=True)
STORY_PATH = PROJECT / 'stories/frutalandia/contador-do-butico-ep01.json'
assert CENAS in ('all','s01','s02','s03','s04','s05','s06')
assert MOTOR_3D in ('WORKBENCH','CYCLES')
print('Cenas:',CENAS,'| motor:',MOTOR_3D,'| destino:',BASE)

In [ ]:
#@title 2. Baixar código original do GitHub
REPO_URL = 'https://github.com/nutricionistaalmeidavh-spec/moneyPrintTest.git'
BRANCH = 'feat/frutalandia-cenas-paralelas-contador'
if not (PROJECT / '.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch',BRANCH,REPO_URL,str(PROJECT)],check=True,timeout=240)
else:
    subprocess.run(['git','-C',str(PROJECT),'pull','--ff-only','origin',BRANCH],check=True,timeout=240)
sys.path.insert(0,str(PROJECT))
os.chdir(PROJECT)
from scripts.novelinha_frutas import validate_story
from scripts.frutalandia_scenes import probe_mp4, scene_ids, get_scene, assemble
story=validate_story(json.loads(STORY_PATH.read_text(encoding='utf-8')))
assert story['format']['target_seconds']==48
assert scene_ids(story)==['s01','s02','s03','s04','s05','s06']
print('Roteiro validado:',story['title'])

In [ ]:
#@title 3. Instalar Blender, FFmpeg, Xvfb e Piper open source
# Este passo usa a internet do Colab, NÃO do Kaggle.
need_binaries=['blender','ffmpeg','ffprobe','xvfb-run','xauth']
missing=[x for x in need_binaries if shutil.which(x) is None]
if missing:
    subprocess.run(['apt-get','update','-qq'],check=True,timeout=240)
    subprocess.run(['apt-get','install','-y','-qq','blender','ffmpeg','xvfb','xauth','libgl1-mesa-dri','fonts-dejavu-core'],check=True,timeout=450)
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check','piper-tts==1.8.0'],check=True,timeout=300)
for binary in need_binaries+['piper']:
    assert shutil.which(binary), f'Dependência ausente: {binary}'
from scripts.frutalandia_episode_runner import fetch_piper
model=fetch_piper(Path('/tmp/frutalandia-piper-model'))
assert model.is_file() and model.stat().st_size>60_000_000
print('Piper pt_BR-faber-medium validado com SHA-256; Blender/FFmpeg disponíveis.')
print('GPU opcional; uso da GPU não é garantido pelo Google Colab.')

In [ ]:
#@title 4. Renderizar as cenas escolhidas (preserva cenas prontas)
# Uma cena inválida não é tratada como pronta. O arquivo MP4 precisa passar no ffprobe.
selected=scene_ids(story) if CENAS=='all' else [CENAS]
for sid in selected:
    expected=get_scene(story,sid)['seconds']
    clip=CLIPS/('scene-'+sid+'.mp4')
    if clip.exists():
        try:
            print('REAPROVEITANDO',sid,probe_mp4(clip,expected))
            continue
        except Exception:
            clip.unlink()
    working=RAW/sid
    working.mkdir(parents=True,exist_ok=True)
    env=os.environ.copy()
    env['FRUTALANDIA_RENDER_ENGINE']=MOTOR_3D
    env['FRUTALANDIA_CPU_SAMPLES']='1'
    env['LIBGL_ALWAYS_SOFTWARE']='1' if MOTOR_3D=='WORKBENCH' else env.get('LIBGL_ALWAYS_SOFTWARE','0')
    env['PYTHONUNBUFFERED']='1'
    cmd=['xvfb-run','-a','-s','-screen 0 1280x800x24',sys.executable,'-u',str(PROJECT/'scripts/frutalandia_episode_runner.py'),'--story',str(STORY_PATH),'--output',str(working),'--scene-id',sid]
    print('INICIANDO CENA',sid,'—',expected,'segundos; logs em',working/'render.log',flush=True)
    log=working/'render.log'
    with log.open('w',encoding='utf-8') as fh:
        try:
            result=subprocess.run(cmd,cwd=PROJECT,env=env,stdout=fh,stderr=subprocess.STDOUT,timeout=2400)
        except subprocess.TimeoutExpired as err:
            print('TEMPO ESGOTADO em',sid,'; restante preservado.',flush=True)
            raise RuntimeError('Render da cena ultrapassou 40 minutos; confira render.log') from err
    if result.returncode:
        print('ERRO REAL EM',sid,'— últimas linhas do render:',flush=True)
        print('\n'.join(log.read_text(encoding='utf-8',errors='replace').splitlines()[-35:]))
        raise RuntimeError('Render falhou na cena '+sid+'; anteriores preservadas em '+str(CLIPS))
    rendered=working/('scene-'+sid+'.mp4')
    assert rendered.is_file(), 'Render terminou sem gerar '+rendered.name
    print('VALIDADO',sid,probe_mp4(rendered,expected))
    shutil.copy2(rendered,clip)
    print('CENA CONCLUÍDA',sid,'->',clip,flush=True)
print('Cenas atualmente disponíveis:',[p.name for p in sorted(CLIPS.glob('scene-*.mp4'))])

In [ ]:
#@title 5. Montar 48 s somente se TODAS as seis cenas estiverem validadas
missing=[sid for sid in scene_ids(story) if not (CLIPS/('scene-'+sid+'.mp4')).is_file()]
if missing:
    print('Ainda faltam cenas:',missing,'. Execute essas cenas na célula 4 antes da montagem.')
else:
    for sid in scene_ids(story):
        print('Conferindo',sid,probe_mp4(CLIPS/('scene-'+sid+'.mp4'),8))
    output=assemble(story,CLIPS,FINAL)
    destination=FINAL/'frutalandia-contador-ep01.mp4'
    if output != destination:
        output.replace(destination)
    meta_file=FINAL/'manifest.json'
    meta=json.loads(meta_file.read_text(encoding='utf-8'))
    meta['video_file']=destination.name
    meta_file.write_text(json.dumps(meta,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    print('EPISÓDIO COMPLETO',destination,'—',probe_mp4(destination,48))

In [ ]:
#@title 6. Baixar MP4 ou salvar as cenas já produzidas
from google.colab import files
final_video=FINAL/'frutalandia-contador-ep01.mp4'
if final_video.is_file():
    print('MP4 VERIFICADO:',probe_mp4(final_video,48))
    if BAIXAR_AUTOMATICAMENTE: files.download(str(final_video))
else:
    partials=sorted(CLIPS.glob('scene-*.mp4'))
    if partials:
        archive=BASE/'cenas-parciais-butico.zip'
        with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_STORED) as z:
            for p in partials: z.write(p,arcname=p.name)
        print('Ainda não há episódio completo. Backup de',len(partials),'cenas:',archive)
        if BAIXAR_AUTOMATICAMENTE: files.download(str(archive))
    else:
        print('Nenhuma cena validada disponível para baixar.')

### Limitações verificáveis

- **Workbench:** animação 3D de baixo custo visual, sem efeitos cinematográficos do Cycles. Se houver falha no OpenGL/Xvfb da sessão, o notebook mostrará o erro; não inventa MP4.
- **Piper:** voz gratuita em português brasileiro, mas **não garante sotaque nordestino autêntico**; o roteiro contém regionalismos.
- **Colab gratuito:** depende da disponibilidade de sessões, recursos e conectividade da sua conta. A GPU T4 é opcional, e a sessão pode ser desconectada.
- **Não publica vídeos**, não pede token, não mexe em Secrets e não usa APIs pagas.
- **Persistência:** os arquivos estão em `/content`, que é temporário. Baixe os resultados antes de encerrar a sessão.